# RAV-Med — Step 1 (7B re-run): Downstream Answer-Quality Test

**Question this notebook answers:** does verifying retrieved evidence with ERS *before generation* improve what an open LLM actually answers?

It reuses the **exact 1,000 queries and the exact CLEAN / DRIFT / NOISE / CONTRADICTION passages** from your N = 1000 run (`rav_med_full_results.csv`), so nothing is rebuilt and the numbers tie to the paper.

**Settings compared (per query):**

| Setting | Evidence given to the LLM |
|---|---|
| `NO_RETRIEVAL` | none (closed-book reference) |
| `CLEAN` | the true abstract |
| `INJ_<X>` | true abstract + one corrupted passage X ∈ {DRIFT, NOISE, CONTRADICTION} |
| `MIXED` | true abstract + all three corrupted passages |
| `…+FILTER` | the same sets after RAV-Med removes passages with ERS < T |

**Two filters are reported, because of the claim-relative issue in the paper:**
* **Deployable (`proxy`)** — at deployment there is no gold claim. The claim is a *label-free* declarative rewrite of the question, produced by the same LLM. This is the number that goes in the main results.
* **Oracle (`oracle`)** — uses the PubMedQA gold conclusion sentence as the claim. It leaks the answer, so it is **an upper bound only** and must be labelled as such in the paper.

**Outcomes:** answer accuracy and macro-F1 (yes/no/maybe vs. PubMedQA gold), abstention rate, and unsupported-claim rate of the one-sentence justification (judged by a *different* NLI model than the verifier).

## 1. Install

In [ ]:
!pip install -q datasets sentence-transformers transformers accelerate bitsandbytes

## 2. Configuration
Defaults fit a free Colab T4. The LLM is the only thing you may want to change.

In [ ]:
import os, re, json, random, hashlib
import numpy as np, pandas as pd, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

RESULTS_CSV   = "rav_med_full_results.csv"      # from your N=1000 zip
LLM_NAME      = "Qwen/Qwen2.5-7B-Instruct"
LOAD_4BIT     = False   # False on L4/A100 (fp16, ~15 GB). Set True on a free T4 (4-bit, slower but fits)
JUDGE_NLI     = "microsoft/deberta-large-mnli"  # judge for unsupported claims (different family from the verifier)
EMBED_MODEL   = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
NLI_MODEL     = "roberta-large-mnli"

N_QUERIES     = 1000          # set to 200 for a quick smoke test first
FILTER_T      = [0.2]         # primary threshold only (fixed in advance); halves the work vs. the 3B run
PRIMARY_T     = 0.2
MAX_WORDS_PER_PASSAGE = 350
GEN_BATCH     = 8
MAX_NEW_TOKENS = 60
DEVICE = "cuda"
# Cache lives on Google Drive so a runtime reset cannot lose hours of generation.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/rav_med", exist_ok=True)
    CACHE_PATH = "/content/drive/MyDrive/rav_med/llm_cache_7b.json"
except Exception as e:
    print("Drive not available, using local cache:", e); CACHE_PATH = "llm_cache_7b.json"

## 3. Load the existing N=1000 conditions and attach the gold labels

In [ ]:
if not os.path.exists(RESULTS_CSV):
    from google.colab import files
    print("Upload rav_med_full_results.csv (from rav_med_outputs.zip)")
    files.upload()

df = pd.read_csv(RESULTS_CSV)
assert set(df.condition) == {"CLEAN", "DRIFT", "CONTRADICTION", "NOISE"}

from datasets import load_dataset
pqa = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
decision = {ex["pubid"]: ex["final_decision"] for ex in pqa}

passages = {c: df[df.condition == c].set_index("pubid")["passage"] for c in ["CLEAN","DRIFT","CONTRADICTION","NOISE"]}
base = df[df.condition == "CLEAN"].set_index("pubid")[["question", "claim"]]
Q = base.copy()
Q["gold"] = [decision[p] for p in Q.index]
Q = Q.iloc[:N_QUERIES].copy()          # same order as the main run
Q.index.name = "pubid"
print(len(Q), "queries"); print(Q.gold.value_counts(normalize=True).round(3))
print("Majority-class accuracy:", round(Q.gold.value_counts(normalize=True).max(), 3))

## 4. Load the LLM and the generation helper

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
if tok.pad_token is None: tok.pad_token = tok.eos_token
if LOAD_4BIT:
    from transformers import BitsAndBytesConfig
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, device_map={"": 0}, quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")).eval()
else:
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16).to(DEVICE).eval()

cache = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {}
def _key(system, user): return hashlib.md5((LLM_NAME + "\x00" + system + "\x00" + user).encode()).hexdigest()

def _try(chunk, mnt):
    try:
        texts = [tok.apply_chat_template([{"role":"system","content":s},{"role":"user","content":u}],
                 tokenize=False, add_generation_prompt=True) for _, (s, u) in chunk]
        enc = tok(texts, return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad():
            out = llm.generate(**enc, max_new_tokens=mnt, do_sample=False, pad_token_id=tok.pad_token_id)
        return [d.strip() for d in tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    except torch.cuda.OutOfMemoryError:
        return None

def _run_chunk(chunk, mnt):
    r = _try(chunk, mnt)
    if r is not None: return r
    import gc; gc.collect(); torch.cuda.empty_cache()
    if len(chunk) == 1:
        raise RuntimeError("A single prompt does not fit; lower MAX_WORDS_PER_PASSAGE to 250 and re-run.")
    h = len(chunk) // 2
    return _run_chunk(chunk[:h], mnt) + _run_chunk(chunk[h:], mnt)

def generate_many(pairs, max_new_tokens=MAX_NEW_TOKENS, batch=GEN_BATCH, tag=""):
    """pairs: list of (system, user). Cached on disk; halves the batch automatically on CUDA OOM."""
    todo = {}
    for s, u in pairs:
        k = _key(s, u)
        if k not in cache: todo[k] = (s, u)
    items = sorted(todo.items(), key=lambda kv: len(kv[1][1]))
    print(f"{tag}: {len(pairs)} prompts, {len(items)} new to generate")
    for b in range(0, len(items), batch):
        chunk = items[b:b+batch]
        for (k, _), o in zip(chunk, _run_chunk(chunk, max_new_tokens)): cache[k] = o
        if (b // batch) % 25 == 0:
            json.dump(cache, open(CACHE_PATH, "w")); print(f"  {b+len(chunk)}/{len(items)}", end="\r")
    json.dump(cache, open(CACHE_PATH, "w"))
    return [cache[_key(s, u)] for s, u in pairs]


## 5. Label-free claim for the deployable filter
The LLM rewrites each *question* as an affirmative declarative statement. It never sees the gold answer or the abstract.

In [ ]:
HYP_SYS = "You rewrite biomedical research questions as statements. Output only the statement."
def hyp_prompt(q):
    return ("Rewrite this yes/no research question as ONE affirmative declarative sentence that asserts "
            "the proposition being asked about. Do not answer the question or add information.\n\n"
            f"Question: {q}\nStatement:")
outs = generate_many([(HYP_SYS, hyp_prompt(q)) for q in Q.question], max_new_tokens=60, tag="hypotheses")
Q["hyp"] = [o.split("\n")[0].strip() for o in outs]
for _, r in Q.head(8).iterrows():
    print("Q   :", r.question); print("HYP :", r.hyp); print("GOLD:", r.claim, "\n")

## 6. Score every passage against both claims
* **oracle ERS** — already in your CSV (gold-claim based).
* **proxy ERS** — recomputed against the label-free hypothesis, same SBERT and RoBERTa-Large-MNLI (max_length 256), α = β = 1.

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
nli = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(DEVICE).eval()
c_idx = [i for i, l in nli.config.id2label.items() if l.lower().startswith("contra")][0]

@torch.no_grad()
def contra(prem, hyp, bs=32):
    out = []
    for i in range(0, len(prem), bs):
        enc = nli_tok(prem[i:i+bs], hyp[i:i+bs], truncation=True, padding=True, max_length=256, return_tensors="pt").to(DEVICE)
        out += torch.softmax(nli(**enc).logits, -1)[:, c_idx].cpu().tolist()
    return np.array(out)
def cos(a, b, bs=32):
    A = embedder.encode(a, batch_size=bs, convert_to_numpy=True, normalize_embeddings=True)
    B = embedder.encode(b, batch_size=bs, convert_to_numpy=True, normalize_embeddings=True)
    return (A * B).sum(1)

CONDS = ["CLEAN", "DRIFT", "NOISE", "CONTRADICTION"]
ers_proxy, ers_oracle = {}, {}
orc = df.set_index(["pubid", "condition"])["ERS"]
for c in CONDS:
    P = [passages[c][p] for p in Q.index]
    H = Q.hyp.tolist()
    ers_proxy[c]  = pd.Series(cos(P, H) - contra(P, H), index=Q.index)
    ers_oracle[c] = pd.Series([orc[(p, c)] for p in Q.index], index=Q.index)

diag = pd.DataFrame({c: [ers_proxy[c].mean(), ers_oracle[c].mean()] for c in CONDS}, index=["mean ERS (proxy claim)", "mean ERS (oracle claim)"])
print(diag.round(3))
print("\nProxy-claim mean ERS of the CLEAN passage, by gold answer (expect 'no' to be lower - the claim-relative effect):")
print(ers_proxy["CLEAN"].groupby(Q.gold).mean().round(3))

## 7. Build the evidence sets for every setting
Passage order inside a prompt is a fixed per-query shuffle (seeded by pubid), so a filtered set is an order-preserving subset of the unfiltered one and identical prompts are generated only once.

In [ ]:
def order_for(pubid):
    rng = random.Random(f"{SEED}-{pubid}")
    o = CONDS[:]; rng.shuffle(o); return o

SETS = {"CLEAN": ["CLEAN"], "INJ_DRIFT": ["CLEAN","DRIFT"], "INJ_NOISE": ["CLEAN","NOISE"],
        "INJ_CONTRADICTION": ["CLEAN","CONTRADICTION"], "MIXED": CONDS[:]}

def trunc(t, n=MAX_WORDS_PER_PASSAGE):
    w = t.split(); return t if len(w) <= n else " ".join(w[:n]) + " ..."

def kept(pubid, members, mode, T):
    """Conditions surviving the filter. mode in {None,'proxy','oracle'}."""
    order = [c for c in order_for(pubid) if c in members]
    if mode is None: return order
    s = ers_proxy if mode == "proxy" else ers_oracle
    return [c for c in order if s[c][pubid] >= T]

SYS_RAG = ("You are a careful biomedical question-answering assistant. Use ONLY the evidence passages provided. "
           "If they are insufficient or conflict in a way you cannot resolve, answer maybe.")
SYS_CB  = "You are a careful biomedical question-answering assistant. No evidence is provided; use your own knowledge."
FMT = ("Answer with exactly two lines:\nAnswer: <yes|no|maybe>\nReason: <one sentence>")

def make_prompt(pubid, question, conds):
    if conds is None:
        return SYS_CB, f"Question: {question}\n\n{FMT}"
    ev = "\n\n".join(f"[Passage {i+1}] {trunc(passages[c][pubid])}" for i, c in enumerate(conds))
    return SYS_RAG, f"Evidence:\n{ev}\n\nQuestion: {question}\n\n{FMT}"

# setting name -> {pubid: list of kept conditions or None for closed-book}
settings = {"NO_RETRIEVAL": {p: None for p in Q.index}}
for name, members in SETS.items():
    settings[name] = {p: kept(p, members, None, None) for p in Q.index}
    if name == "CLEAN": continue
    for mode in ["proxy", "oracle"]:
        for T in FILTER_T:
            settings[f"{name}+FILTER_{mode}_T{T}"] = {p: kept(p, members, mode, T) for p in Q.index}

work = {}   # (setting, pubid) -> (system,user) or None if abstained (no passage survived)
for sname, m in settings.items():
    for p, conds in m.items():
        if conds is not None and len(conds) == 0: work[(sname, p)] = None
        else: work[(sname, p)] = make_prompt(p, Q.question[p], conds)
uniq = {v for v in work.values() if v is not None}
print(len(settings), "settings;", len(uniq), "unique prompts to generate")

## 8. Generate answers (resumable; run again after a disconnect and it continues from the cache)

In [ ]:
import gc
embedder.to("cpu"); nli.to("cpu"); gc.collect(); torch.cuda.empty_cache()   # free GPU memory for long prompts
uniq_list = sorted(uniq)
_ = generate_many(uniq_list, tag="answers")
gen = {v: cache[_key(*v)] for v in uniq_list}

## 9. Parse and score accuracy

In [ ]:
def parse(txt):
    # tolerant: accepts "Answer: yes", "Yes", "**No**.", with the reason on a later line
    a = re.search(r"^\W*(?:answer\s*:\s*)?\W*(yes|no|maybe)\b", txt.strip(), re.I)
    r = re.search(r"reason\s*:\s*(.+)", txt, re.I | re.S)
    return (a.group(1).lower() if a else None), (r.group(1).strip().split("\n")[0] if r else "")

rows = []
for (sname, p), v in work.items():
    if v is None:
        rows.append(dict(setting=sname, pubid=p, pred="maybe", reason="", abstained=True, parsed=True))
    else:
        a, r = parse(gen[v])
        rows.append(dict(setting=sname, pubid=p, pred=a or "maybe", reason=r, abstained=False, parsed=a is not None))
R = pd.DataFrame(rows)
R["gold"] = R.pubid.map(Q.gold)
R["correct"] = (R.pred == R.gold).astype(int)
print("Unparseable outputs (counted as 'maybe'):", f"{(~R.parsed).mean():.2%}")

In [ ]:
from sklearn.metrics import f1_score
rng = np.random.default_rng(SEED)
pubids = list(Q.index); n = len(pubids)
M = R.pivot(index="pubid", columns="setting", values="correct").loc[pubids]
AB = R.pivot(index="pubid", columns="setting", values="abstained").loc[pubids]
boot_idx = rng.integers(0, n, size=(2000, n))          # query-level cluster bootstrap

def acc_ci(col):
    v = M[col].values; b = v[boot_idx].mean(1)
    return v.mean(), *np.percentile(b, [2.5, 97.5])
def diff_ci(a, b_):
    d = (M[a] - M[b_]).values; bs = d[boot_idx].mean(1)
    return d.mean(), *np.percentile(bs, [2.5, 97.5])

rows = []
for s in M.columns:
    sub = R[R.setting == s].set_index("pubid").loc[pubids]
    m, lo, hi = acc_ci(s)
    rows.append(dict(setting=s, accuracy=m, ci_lo=lo, ci_hi=hi,
                     macro_F1=f1_score(sub.gold, sub.pred, average="macro", labels=["yes","no","maybe"]),
                     abstain_rate=AB[s].mean()))
T_ACC = pd.DataFrame(rows).set_index("setting")
order_cols = ["NO_RETRIEVAL","CLEAN"] + [c for c in T_ACC.index if c not in ("NO_RETRIEVAL","CLEAN")]
T_ACC = T_ACC.loc[order_cols]
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 100)
print(T_ACC.round(3))
T_ACC.to_csv("step1_table_accuracy.csv")

## 10. Does the filter help? Paired differences with query-level bootstrap CIs
* **Recovery** = filtered − unfiltered, within the same corruption (positive = the filter helped).
* **Cost** = CLEAN-only − filtered-on-clean-evidence is not separable here, so the harm of over-filtering shows up as the gap to `CLEAN`.

In [ ]:
rows = []
for name in ["INJ_DRIFT","INJ_NOISE","INJ_CONTRADICTION","MIXED"]:
    rows.append(dict(contrast=f"{name}: unfiltered vs CLEAN (damage)", **dict(zip(["diff","lo","hi"], diff_ci(name, "CLEAN")))))
    for mode in ["proxy","oracle"]:
        for T in FILTER_T:
            f = f"{name}+FILTER_{mode}_T{T}"
            rows.append(dict(contrast=f"{name}: {mode} filter T={T} minus unfiltered (recovery)", **dict(zip(["diff","lo","hi"], diff_ci(f, name)))))
            rows.append(dict(contrast=f"{name}: {mode} filter T={T} minus CLEAN (residual gap)", **dict(zip(["diff","lo","hi"], diff_ci(f, "CLEAN")))))
T_DIFF = pd.DataFrame(rows); print(T_DIFF.round(3).to_string(index=False))
T_DIFF.to_csv("step1_table_paired_differences.csv", index=False)

## 11. Stratified by gold answer (exposes the claim-relative limitation)

In [ ]:
strata = []
for s in ["CLEAN","MIXED"] + [f"MIXED+FILTER_{m}_T{PRIMARY_T}" for m in ["proxy","oracle"]]:
    sub = R[R.setting == s]
    g = sub.groupby("gold").agg(n=("correct","size"), accuracy=("correct","mean"), abstain=("abstained","mean"))
    g.insert(0, "setting", s); strata.append(g.reset_index())
T_STRAT = pd.concat(strata); print(T_STRAT.round(3).to_string(index=False))
T_STRAT.to_csv("step1_table_by_gold.csv", index=False)

print("\nShare of passages KEPT by the primary-T filter, by condition and gold answer:")
kept_rows = []
for mode, sc in [("proxy", ers_proxy), ("oracle", ers_oracle)]:
    for c in CONDS:
        kk = (sc[c] >= PRIMARY_T).groupby(Q.gold).mean()
        kept_rows.append(dict(mode=mode, condition=c, **kk.round(3).to_dict(), overall=round((sc[c] >= PRIMARY_T).mean(), 3)))
T_KEPT = pd.DataFrame(kept_rows); print(T_KEPT.to_string(index=False)); T_KEPT.to_csv("step1_table_kept_share.csv", index=False)

## 12. Unsupported-claim rate of the justification
Judge: `microsoft/deberta-large-mnli` (a different model family from the RoBERTa verifier, to avoid grading the verifier with itself).
Premise = the **CLEAN abstract** (the ground-truth evidence); hypothesis = the model's one-sentence reason.
* **Contradicted** = P(contradiction) > 0.5. **Unsupported** = P(entailment) < 0.5.
This is an automatic proxy; Step 3 (human validation) will check it on a sample.

In [ ]:
del nli; torch.cuda.empty_cache()
j_tok = AutoTokenizer.from_pretrained(JUDGE_NLI)
judge = AutoModelForSequenceClassification.from_pretrained(JUDGE_NLI).to(DEVICE).eval()
lab = {l.lower(): i for i, l in judge.config.id2label.items()}
iE = [i for l, i in lab.items() if l.startswith("entail")][0]; iC = [i for l, i in lab.items() if l.startswith("contra")][0]

need = R[(~R.abstained) & (R.reason.str.len() > 0)][["pubid", "reason"]].drop_duplicates()
print(len(need), "unique (query, reason) pairs to judge")
need_list = list(need.itertuples(index=False))
res = {}
with torch.no_grad():
    for i in range(0, len(need_list), 16):
        ch = need_list[i:i+16]
        enc = j_tok([passages["CLEAN"][p] for p, _ in ch], [r for _, r in ch], truncation="only_first",
                    padding=True, max_length=512, return_tensors="pt").to(DEVICE)
        pr = torch.softmax(judge(**enc).logits, -1).cpu().numpy()
        for (p, r), x in zip(ch, pr): res[(p, r)] = (x[iE], x[iC])
R["p_ent"] = [res.get((p, r), (np.nan, np.nan))[0] for p, r in zip(R.pubid, R.reason)]
R["p_con"] = [res.get((p, r), (np.nan, np.nan))[1] for p, r in zip(R.pubid, R.reason)]
R["unsupported"] = np.where(R.p_ent.notna(), (R.p_ent < 0.5).astype(float), np.nan)
R["contradicted"] = np.where(R.p_con.notna(), (R.p_con > 0.5).astype(float), np.nan)

U = R.groupby("setting").agg(n_judged=("unsupported", "count"), unsupported=("unsupported", "mean"), contradicted=("contradicted", "mean"))
U = U.loc[[s for s in order_cols if s in U.index]]
print(U.round(3)); U.to_csv("step1_table_unsupported.csv")

## 13. Figure and save everything

In [ ]:
import matplotlib.pyplot as plt
show = ["NO_RETRIEVAL","CLEAN","MIXED",f"MIXED+FILTER_proxy_T{PRIMARY_T}",f"MIXED+FILTER_oracle_T{PRIMARY_T}"]
t = T_ACC.loc[show]
plt.figure(figsize=(7.5, 4))
plt.bar(range(len(show)), t.accuracy, yerr=[t.accuracy - t.ci_lo, t.ci_hi - t.accuracy], capsize=4)
plt.xticks(range(len(show)), ["No\nretrieval","Clean","Mixed\n(unfiltered)","Mixed +\nRAV-Med\n(deployable)","Mixed +\nRAV-Med\n(oracle, upper bd.)"], fontsize=8)
plt.axhline(Q.gold.value_counts(normalize=True).max(), ls="--", c="gray", lw=0.8); plt.text(4.45, Q.gold.value_counts(normalize=True).max()+0.005, "majority class", ha="right", fontsize=7)
plt.ylabel("PubMedQA accuracy"); plt.title(f"Downstream answer accuracy ({LLM_NAME.split('/')[-1]}, n={n})")
plt.tight_layout(); plt.savefig("step1_fig_accuracy.png", dpi=150); plt.show()

R.to_csv("step1_all_predictions.csv", index=False)
Q.to_csv("step1_queries_with_hypotheses.csv")
import shutil
os.makedirs("rav_med_step1_7b_outputs", exist_ok=True)
for f in ["step1_table_accuracy.csv","step1_table_paired_differences.csv","step1_table_by_gold.csv","step1_table_kept_share.csv",
          "step1_table_unsupported.csv","step1_fig_accuracy.png","step1_all_predictions.csv","step1_queries_with_hypotheses.csv", CACHE_PATH]:
    if os.path.exists(f): shutil.copy(f, "rav_med_step1_7b_outputs")
shutil.make_archive("rav_med_step1_7b_outputs", "zip", "rav_med_step1_7b_outputs")
try:
    from google.colab import files; files.download("rav_med_step1_7b_outputs.zip")
except ImportError: pass
print("Saved rav_med_step1_7b_outputs.zip")